<a href="https://colab.research.google.com/github/Jameslyo69/informe_sem7/blob/main/S7_ETL_Y_DATA_LAKE_EN_GOOGLE_COLAB_ipynb_copia_(5)_copia_txt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 7: Pipeline ETL, Profiling y Gobernanza en el Data Lake
**Materia:** Fundamentos de Big Data (IDSD413) - EPN

**Objetivo:** Simular un escenario real de ingesta de datos ruidosos (anomalías, nulos, duplicados), ejecutar un perfilamiento estadístico para dimensionar el caos, y procesarlos a través de un flujo ETL construyendo las capas lógicas de un Data Lake (Capa *Raw* y Capa *Curated*).



In [ ]:
import pandas as pd
import numpy as np
import os

print("1/3 Generando datos base perfectos en memoria...")
np.random.seed(42)
N_ROWS = 5_000_000
df_base = pd.DataFrame({
    'id_transaccion': np.arange(N_ROWS),
    'id_cliente': np.random.randint(1, 100000, N_ROWS),
    'monto': np.random.uniform(5.0, 2500.0, N_ROWS),
    'categoria': np.random.choice(['Electronica', 'Ropa', 'Hogar', 'Alimentos', 'Juguetes'], N_ROWS),
    'estado': np.random.choice(['Completado', 'Pendiente', 'Cancelado', 'Reembolsado'], N_ROWS)
})

# Guardamos una base fresca para evitar el error de bytes corruptos en Colab
nombre_fresco = 'base_fresca.parquet'
df_base.to_parquet(nombre_fresco, engine='pyarrow')

print("2/3 Leyendo archivo desde disco e inyectando caos (Simulación de ingesta real)...")
df_original = pd.read_parquet(nombre_fresco, engine='pyarrow')

# a) Inyectamos un 5% de nulos
n_nulos = int(len(df_original) * 0.05)
indices_nulos = np.random.choice(df_original.index, n_nulos, replace=False)
df_original.loc[indices_nulos, 'monto'] = np.nan

# b) Duplicamos 50,000 registros
df_duplicados = df_original.sample(n=50000)
df_ingesta = pd.concat([df_original, df_duplicados], ignore_index=True)

# c) Inyectamos outliers (valores absurdos que rompen la estadística)
df_ingesta.loc[0:100, 'monto'] = 9999999.99

print(f"3/3 ¡Éxito! Dataset de ingesta listo. Total de filas: {len(df_ingesta):,}")

1/3 Generando datos base perfectos en memoria...
2/3 Leyendo archivo desde disco e inyectando caos (Simulación de ingesta real)...
3/3 ¡Éxito! Dataset de ingesta listo. Total de filas: 5,050,000


## Fase 1: Profiling y Capa RAW (El Diagnóstico del Caos)
Antes de transformar, debemos perfilar el archivo crudo para entender la magnitud del problema y luego guardarlo en la capa inmutable del Data Lake.

In [ ]:
print("--- DIAGNÓSTICO ESTADÍSTICO (PROFILING) ---\n")

# 1. Auditoría de Nulos y Duplicados
print("1. Valores Nulos por Columna:")
print(df_ingesta.isnull().sum())
print(f"\n2. Registros Duplicados Exactos: {df_ingesta.duplicated().sum():,}")

# 2. Auditoría de Outliers
print("\n3. Resumen Estadístico de 'monto':")
print(df_ingesta['monto'].describe().apply(lambda x: f"{x:,.2f}"))

# 3. NUEVO (soporte Reto 2): ¿en qué estados están los nulos?
print("\n4. Distribución de nulos de 'monto' por estado:")
print(df_ingesta[df_ingesta['monto'].isnull()]['estado'].value_counts())

# 4. NUEVO (Reto 1): Arquitectura del Data Lake con 3 capas
CAPAS = ['raw', 'curated', 'quarantine']
for capa in CAPAS:
    os.makedirs(f'./datalake/{capa}', exist_ok=True)
print(f"\nCapas del Data Lake creadas: {CAPAS}")

# 5. Guardado en Capa RAW (Inmutable)
print("\n--- EXTRACT & LOAD (EL) ---")
df_ingesta.to_parquet('./datalake/raw/ingesta_transacciones_sucio.parquet')
print("¡Archivo RAW almacenado con éxito! Listos para limpiar.")

--- DIAGNÓSTICO ESTADÍSTICO (PROFILING) ---

1. Valores Nulos por Columna:
id_transaccion         0
id_cliente             0
monto             252510
categoria              0
estado                 0
dtype: int64

2. Registros Duplicados Exactos: 49,999

3. Resumen Estadístico de 'monto':
count    4,797,490.00
mean         1,463.00
std         45,882.62
min              5.00
25%            628.91
50%          1,252.56
75%          1,876.37
max      9,999,999.99
Name: monto, dtype: object

4. Distribución de nulos de 'monto' por estado:
estado
Pendiente      63493
Cancelado      63176
Completado     63049
Reembolsado    62792
Name: count, dtype: int64

Capas del Data Lake creadas: ['raw', 'curated', 'quarantine']

--- EXTRACT & LOAD (EL) ---
¡Archivo RAW almacenado con éxito! Listos para limpiar.


## Fase 2: Transform (T) - Pipeline ETL de Limpieza y Normalización
Levantamos el dato desde RAW y ejecutamos limpieza masiva: eliminar duplicados, tratar nulos mediante imputación robusta (mediana) y eliminar valores atípicos (outliers) utilizando el Rango Intercuartílico (IQR). Finalmente, normalizamos para Machine Learning.

### Reto 2: Descarte inteligente vs. imputación
Una transacción **Cancelada** sin monto no aporta valor predictivo: imputarle una mediana
inventaría una venta que nunca ocurrió y sesgaría el modelo. Por eso se elimina.
Para Completado, Pendiente y Reembolsado se mantiene la imputación por mediana por categoría.

In [ ]:
print("Iniciando Transformación...\n")

# 1. Eliminación de Duplicados
antes = len(df_ingesta)
df_limpio = df_ingesta.drop_duplicates().copy()
print(f"Duplicados eliminados: {antes - len(df_limpio):,}")

# 2. RETO 2 - Descarte inteligente: Cancelado + monto nulo
mask_cancelado_nulo = (df_limpio['estado'] == 'Cancelado') & (df_limpio['monto'].isnull())
n_descartados = mask_cancelado_nulo.sum()
df_limpio = df_limpio[~mask_cancelado_nulo].copy()
print(f"[Reto 2] Filas 'Cancelado' con monto nulo ELIMINADAS: {n_descartados:,}")

# 3. RETO 2 - Imputación por mediana para el resto de estados
print("\nNulos pendientes de imputar por estado:")
print(df_limpio[df_limpio['monto'].isnull()]['estado'].value_counts())

n_imputados = df_limpio['monto'].isnull().sum()
medianas = df_limpio.groupby('categoria')['monto'].*transform('median')
df_limpio['monto'] = df_limpio['monto'].fillna(medianas)
print(f"\n[Reto 2] Nulos IMPUTADOS con mediana por categoría: {n_imputados:,}")

print("\nMedianas utilizadas por categoría:")
print(df_limpio.groupby('categoria')['monto'].median().round(2))

# 4. Validación de calidad
assert df_limpio['monto'].isnull().sum() == 0, "Aún existen nulos"
print(f"\nNulos restantes en monto: {df_limpio['monto'].isnull().sum()}  ✔")

Iniciando Transformación...

Duplicados eliminados: 49,999
[Reto 2] Filas 'Cancelado' con monto nulo ELIMINADAS: 62,513

Nulos pendientes de imputar por estado:
estado
Pendiente      62847
Completado     62459
Reembolsado    62173
Name: count, dtype: int64

[Reto 2] Nulos IMPUTADOS con mediana por categoría: 187,479

Medianas utilizadas por categoría:
categoria
Alimentos      1254.44
Electronica    1252.56
Hogar          1252.53
Juguetes       1251.91
Ropa           1251.38
Name: monto, dtype: float64

Nulos restantes en monto: 0  ✔


### Reto 1: Capa de Cuarentena
Los outliers no se eliminan a ciegas: pueden ser fraude o errores críticos del sistema origen.
Se aíslan con trazabilidad (motivo, límites IQR y fecha) en `./datalake/quarantine`
para su auditoría posterior, y solo los datos válidos continúan hacia Curated.

In [9]:
# RETO 1 - Detección de outliers con la regla IQR
Q1 = df_limpio['monto'].quantile(0.25)
Q3 = df_limpio['monto'].quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR
print(f"Q1={Q1:,.2f} | Q3={Q3:,.2f} | IQR={IQR:,.2f}")
print(f"Rango válido: [{limite_inferior:,.2f} ; {limite_superior:,.2f}]")

mask_outlier = (df_limpio['monto'] < limite_inferior) | (df_limpio['monto'] > limite_superior)

# Separar outliers y añadir metadatos de gobernanza
df_cuarentena = df_limpio[mask_outlier].copy()
df_cuarentena['motivo_cuarentena'] = 'OUTLIER_IQR'
df_cuarentena['limite_inferior_iqr'] = limite_inferior
df_cuarentena['limite_superior_iqr'] = limite_superior
df_cuarentena['fecha_cuarentena'] = pd.Timestamp.now()

# Persistir en la capa QUARANTINE (en lugar de borrar)
ruta_cuarentena = './datalake/quarantine/transacciones_outliers.parquet'
df_cuarentena.to_parquet(ruta_cuarentena, index=False)
print(f"\n[Reto 1] Outliers enviados a CUARENTENA: {len(df_cuarentena):,}")
print(f"Archivo: {ruta_cuarentena}")

# Solo los datos válidos continúan el pipeline
total_previo = len(df_limpio)
df_limpio = df_limpio[~mask_outlier].copy()

# Control de integridad: no se pierde ningún registro
assert total_previo == len(df_limpio) + len(df_cuarentena)
print(f"Integridad OK: {total_previo:,} = {len(df_limpio):,} válidos + {len(df_cuarentena):,} en cuarentena ✔")

Q1=653.40 | Q3=1,851.92 | IQR=1,198.51
Rango válido: [-1,144.36 ; 3,649.68]

[Reto 1] Outliers enviados a CUARENTENA: 101
Archivo: ./datalake/quarantine/transacciones_outliers.parquet
Integridad OK: 4,937,488 = 4,937,387 válidos + 101 en cuarentena ✔


## Fase 3: Load (Curated Layer) y Verificación Final
Guardamos el dato limpio en formato columnar en la capa *Curated*. Finalmente, comparamos la estadística del dato curado frente al desastre original.

In [10]:
# 1. Normalización Min-Max (0 a 1) - se mantiene del notebook base
min_monto = df_limpio['monto'].min()
max_monto = df_limpio['monto'].max()
df_limpio['monto_escalado'] = (df_limpio['monto'] - min_monto) / (max_monto - min_monto)
print("Normalización Min-Max completada.")

# 2. Carga en la capa CURATED
df_limpio.to_parquet('./datalake/curated/transacciones_model_ready.parquet', index=False)
print("Datos escritos en la Capa CURATED exitosamente.\n")

# 3. Estadística del dato curado
print("--- ESTADÍSTICA DEL DATO CURADO ---")
print(df_limpio[['monto', 'monto_escalado']].describe().map(lambda x: f"{x:,.2f}"))

print("\n--- MUESTRA LISTA PARA MACHINE LEARNING ---")
display(df_limpio[['id_transaccion', 'categoria', 'estado', 'monto', 'monto_escalado']].head(5))

# 4. Auditoría de la capa QUARANTINE
print("\n--- MUESTRA DE LA CAPA QUARANTINE ---")
display(pd.read_parquet(ruta_cuarentena).head(5))

# 5. Estructura y tamaño del Data Lake
print("\n--- ESTRUCTURA DEL DATA LAKE ---")
for capa in CAPAS:
    ruta = f'./datalake/{capa}'
    for archivo in os.listdir(ruta):
        tam = os.path.getsize(os.path.join(ruta, archivo)) / (1024 * 1024)
        print(f"{capa:<11} → {archivo:<42} {tam:>8.2f} MB")

Normalización Min-Max completada.
Datos escritos en la Capa CURATED exitosamente.

--- ESTADÍSTICA DEL DATO CURADO ---
              monto monto_escalado
count  4,937,387.00   4,937,387.00
mean       1,252.50           0.50
std          706.46           0.28
min            5.00           0.00
25%          653.39           0.26
50%        1,252.53           0.50
75%        1,851.88           0.74
max        2,500.00           1.00

--- MUESTRA LISTA PARA MACHINE LEARNING ---


,id_transaccion,categoria,estado,monto,monto_escalado
101,101,Juguetes,Reembolsado,1015.580233,0.405042
102,102,Alimentos,Pendiente,1837.507104,0.734472
103,103,Electronica,Completado,997.149883,0.397655
104,104,Ropa,Completado,2021.055379,0.808038
105,105,Juguetes,Pendiente,1251.907721,0.499762



--- MUESTRA DE LA CAPA QUARANTINE ---


,id_transaccion,id_cliente,monto,categoria,estado,motivo_cuarentena,limite_inferior_iqr,limite_superior_iqr,fecha_cuarentena
0,0,15796,9999999.99,Alimentos,Cancelado,OUTLIER_IQR,-1144.364192,3649.684122,2026-11-10 00:22:46.168005
1,1,861,9999999.99,Juguetes,Cancelado,OUTLIER_IQR,-1144.364192,3649.684122,2026-11-10 00:22:46.168005
2,2,76821,9999999.99,Ropa,Pendiente,OUTLIER_IQR,-1144.364192,3649.684122,2026-11-10 00:22:46.168005
3,3,54887,9999999.99,Alimentos,Reembolsado,OUTLIER_IQR,-1144.364192,3649.684122,2026-11-10 00:22:46.168005
4,4,6266,9999999.99,Ropa,Pendiente,OUTLIER_IQR,-1144.364192,3649.684122,2026-11-10 00:22:46.168005



--- ESTRUCTURA DEL DATA LAKE ---
raw         → ingesta_transacciones_sucio.parquet           74.41 MB
curated     → transacciones_model_ready.parquet            112.91 MB
quarantine  → transacciones_outliers.parquet                 0.01 MB
